# Proyecto 3: Galaxy Scaling Relations

Santiago Andrés Acosta Díaz

Disclaimer: los menús los hice ayudándome con IA

# Códigos para los menús


## Código para el menú de query

En esta parte, realizamos el código entero para el primer menú, que hace el query desde los datos de la NASA según el tipo de gráficas que deseemos hacer. Para ello, las funciones `action1` y `action2` son las encargadas de hacer el query y guardar el resultado del query en una base de datos de SQLite3.

Este primer menú sólo nos permite elegir los datos necesarios para hacer las gráficas correspondientes. Tiene una opción de filtro, la cual hace que el query sólo returne las filas en donde _todos los datos necesarios_ estén presentes.

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import sqlite3
import pandas as pd

# Define the checkboxes and their associated strings.
# You can assign the same string to multiple checkboxes.
checkbox_data = [
    ("Detection method distribution", ["discoverymethod"]),
    ("Period-radius diagram", ["pl_orbper", "pl_rade"]),
    ("Mass-radius relation", ["pl_bmasse", "pl_rade"]),   # shares string with Option A
    ("Equilibrium temperature histogram", ["pl_eqt"]),
    (r"$R_p$ by host star type", ["pl_rade", "st_spectype"]),   # shares string with Option B
    ("Discovery timeline", ["disc_year", "discoverymethod"]),   # shares string with Option B
]


graphs = []

# Create checkboxes
checkboxes = []
for label, string in checkbox_data:
    cb = widgets.Checkbox(
        value=False,
        description=label,
        indent=False
    )

    checkboxes.append((cb, string, label))

# Container to display the current list
output = widgets.Output()

unique_list = ["pl_name"]

# Function to rebuild the unique list from current checkbox states
def update_list(change=None):
    global unique_list, graphs

    graphs = []
    
    # Count active checkboxes per string
    active_counts = {}
    for cb, items, label in checkboxes:
        for string in items:
            if cb.value:
                active_counts[string] = active_counts.get(string, 0) + 1
        if cb.value:
            graphs.append(label)
    
    # Build the list: include a string only if its count > 0
    unique_list = ["pl_name"] + [s for s, count in active_counts.items() if count > 0]
    
    # Update the output area
    with output:
        clear_output(wait=True)
        print("Current query:", unique_list)


# Attach the update function to every checkbox
for cb, _, _ in checkboxes:
    cb.observe(update_list, names='value')

# Replace the "checkbox_grid" section with this:
row1 = widgets.HBox(
    [checkboxes[0][0], checkboxes[1][0], checkboxes[2][0]],
    layout=widgets.Layout(gap='30px')  # spacing between items
)
row2 = widgets.HBox(
    [checkboxes[3][0], checkboxes[4][0], checkboxes[5][0]],
    layout=widgets.Layout(gap='30px'))




# FILTER BUTTON

# Global variable to store the checkbox state
filterflag = False

# Create the checkbox
extra_checkbox = widgets.Checkbox(
    value=False,
    description="Apply filter",
    indent=False
)

# Observer to update the global variable
def update_global_flag(change):
    global filterflag
    filterflag = change['new']

extra_checkbox.observe(update_global_flag, names='value')


# BOTONES
button1 = widgets.Button(description="Query")
button2 = widgets.Button(description="Save Database")

# TEXTO DE GUARDADO

savepath = ""

# Create the text widget
text_input = widgets.Text(
    value='',
    placeholder='Guardar como...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='300px')   # optional width
)

# Observer: updates the global variable on every keystroke
def on_text_change(change):
    global savepath
    savepath = change['new']

text_input.observe(on_text_change, names='value')



# La base de datos como tal
result = None


# FUNCIONES DE LOS BOTONES

def action1(b):
    
    with output:
        global result
        
        clear_output(wait=True)
        print("Relizando query con parámetros:", unique_list)
        
        if filterflag:
            print("Filtrando filas NULL")
            null_checks = " AND ".join([f"{col} IS NOT NULL" for col in unique_list])
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list, where=null_checks )
        else: 
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list)

        print(f"Resultados del query: {len(result)} resultados")


def action2(b):
    with output:
        global savepath
        
        clear_output(wait=True)
        if savepath == "":
            print("Añadir nombre al archivo")
        else:
            if result == None:
                print("No hay datos")
            else:
                # Convert to Pandas DataFrame
                df = result.to_pandas()
                
                # Create a connection to a SQLite database (it will be created if not exists)
                names = savepath.split(".")

                conn = sqlite3.connect(f'databases/{names[0]}.db')
                
                # Write the DataFrame to a SQLite table (replace 'my_table' with your desired table name)
                df.to_sql('table', conn, if_exists='replace', index=False)
                
                # Close the connection
                conn.close()

                print(f"Se guardó la tabla como {names[0]}.db")
        
button1.on_click(action1)
button2.on_click(action2)




# Third row: empty placeholder in column 1, then the two buttons

row3 = widgets.HBox(
    [extra_checkbox, button1, button2, text_input],
    layout=widgets.Layout(gap='30px')
)

# -------------------------------------------------------------------
# 5. Combine and display
# -------------------------------------------------------------------
ui = widgets.VBox([row1, row2])
ui_save = widgets.VBox([row3, output])

<string>:24: FutureWarning: tag.strict is not set. Currently defaults to False (permissive tag matching). In a future major version the default will change to True (require tags to contain a dot). Set tag.strict = true or tag.strict = false explicitly in your [tool.setuptools_scm] / [tool.vcs-versioning] config to silence this warning.


## Menú de carga de las bases de datos

Lo único interesante es conectarse a la base de datos sin cerrar el `cursor`

In [2]:
# TEXTO DE GUARDADO

conn = None
cursor = None   # El cursos global
loadpath = ""

columns_load = list()
type_load = list()

# Container to display the current list
output_load = widgets.Output()

# Create the text widget
text_input_load = widgets.Text(
    value='',
    placeholder='Cargar archivo...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='300px')   # optional width
)

# Observer: updates the global variable on every keystroke
def on_text_change_load(change):
    global loadpath
    loadpath = change['new']

text_input_load.observe(on_text_change_load, names='value')

# Botón de carga
button_load = widgets.Button(description="Load Database")


# Lista de columnas en la base de datos
column_list = list()

def action2_load(b):
    global loadpath, conn, cursor, columns_load, type_load
    with output_load:

        if cursor is not None:
            conn.close()
        
        clear_output(wait=True)
        if loadpath == "":
            print("Añadir nombre al archivo")
        else:
            try:
                names = loadpath.split(".")
                conn = sqlite3.connect(f'file:databases/{names[0]}.db?mode=rw', uri=True, check_same_thread=False)
                cursor = conn.cursor()
                
                print(f"Se cargó la tabla {names[0]}.db")

                cursor.execute(f"PRAGMA table_info('table')")
                columns = cursor.fetchall()

                # Guardamos info para el filtrado posterior
                columns_load = list()
                type_load = list()
                
                for data in columns:
                    columns_load.append(data[1])
                    type_load.append(data[2])

                print(f"Con datos de {columns_load}")
                    
            except sqlite3.DatabaseError as e:
                print(f"No se puede acceder a la base de datos: {e}")
            except sqlite3.OperationalError as e:
                print(f"No se puede acceder a la base de datos: {e}")
    
button_load.on_click(action2_load)

row_load = widgets.HBox(
    [button_load, text_input_load],
    layout=widgets.Layout(gap='30px')
)

# -------------------------------------------------------------------
# 5. Combine and display
# -------------------------------------------------------------------
ui_load = widgets.VBox([row_load, output_load])

## Código del menú para los filtros personalizados

Donde, por cada tabla cargada, se obtienen sus posibles valores y se aplican los filtros según corresponda

In [60]:
QUERY_PREFIX = "SELECT * FROM 'table'"   
conditions = {}                         
full_query_output = widgets.Output()


# Query global que se va a modificar
QUERY = ""

def rebuild_query():
    """Build and display the full SQL query from all active conditions."""
    global QUERY

    # Query global que se va a modificar
    QUERY = ""
    
    with full_query_output:
        full_query_output.clear_output()
        active = [cond for cond in conditions.values()]
        if active:
            query = QUERY_PREFIX + "\nWHERE " + "AND ".join(active)
        else:
            query = QUERY_PREFIX

        QUERY = query
        print(query)


# Esta función es para crear el menú para los datos de texto
def get_distinct_values(column_name):
    table_name = 'table'
    
    """Return a list of distinct TEXT values from a given column."""
    if cursor is None:
        print("No database loaded.")
        return []
    try:
        
        if column_name == "st_spectype": 
            cursor.execute(f"SELECT DISTINCT SUBSTR({column_name}, 1, 1) FROM '{table_name}' ORDER BY 1")
            return [row[0] for row in cursor.fetchall()[1::]]
            
        elif column_name == "discoverymethod":
            cursor.execute(f"SELECT DISTINCT {column_name} FROM '{table_name}' ORDER BY 1")
            return [row[0] for row in cursor.fetchall()]
        
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return []


# Con esto preparamos los valores para ser displayados en los sliders de rango
def get_min_max(column_name):
    table_name = 'table'
    """Return the min and max numeric values from a given column."""
    if cursor is None:
        print("No database loaded.")
        return None, None
    try:
        # To get rid of the ugly 0 discovery year
        cursor.execute(f"""
            SELECT
                (SELECT MIN({column_name})
                 FROM '{table_name}'
                 WHERE {column_name} > 0) AS min_positive,
                (SELECT MAX({column_name})
                 FROM '{table_name}') AS real_max
        """)
        row = cursor.fetchone()
        if row and row[0] is not None and row[1] is not None:
            return row[0], row[1]
        else:
            return None, None
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None, None
    
        

def create_filters():
    global conditions 

    conditions = {}    
    
    for name, coltype in zip(columns_load, type_load):
        
        if name == "pl_name":
            continue
            
        if coltype == "TEXT":

            # TITLE
            print(f"\t\t\t\t\t\t {name.upper()} \n\n")
            
            options = get_distinct_values(name)
            checkboxes = [widgets.Checkbox(description=opt, value=True) for opt in options]
            
            # Assuming checkboxes is a list of Checkbox widgets
            chunk_size = 4
            chunks = [checkboxes[i:i+chunk_size] for i in range(0, len(checkboxes), chunk_size)]
            checkbox_rows = [widgets.HBox(chunk) for chunk in chunks]
            checkbox_vbox = widgets.VBox(checkbox_rows)

            
            output = widgets.Output()
    
            # Use default args to capture the current checkboxes & output
            def update_string(change, cb_list=checkboxes, out=output, col = name):
                with out:
                    out.clear_output()
                    selected = [cb.description for cb in cb_list if cb.value]
                    
                    quoted = [f"'{opt.replace("'", "''")}'" for opt in selected]

                    if name == "st_spectype":
                        snippet = f"SUBSTR({col}, 1, 1) IN ({', '.join(quoted)})\n"
                    else:
                        snippet = f"{col} IN ({', '.join(quoted)})\n"

                    conditions[col] = snippet           # store for the full query
                    rebuild_query() 
                    
    
            for cb in checkboxes:
                cb.observe(update_string, names='value')
    
            # Show initial state
            update_string(None)
    
            # Display the row immediately, or store and display later
            display(checkbox_vbox, output)
            print("\n\n\n")
            
        else:
            min_val, max_val = get_min_max(name)
            # TITLE
            print(f"\t\t\t\t\t\t {name.upper()} \n\n")
            if min_val is not None and max_val is not None:
                if coltype == "INTEGER":
                    # Integer columns: keep linear IntRangeSlider
                    slider = widgets.IntRangeSlider(
                        value=[min_val, max_val],
                        min=min_val,
                        max=max_val,
                        step=1,
                        description=name,
                        layout=widgets.Layout(width='80%'),
                        continuous_update=False
                    )
                    is_log = False   # flag for the observer
                
                else:  # REAL or FLOAT
                    # Special case: "pl_rade" uses a normal linear FloatRangeSlider
                    if name in ["pl_rade", "pl_eqt"]:
                        slider = widgets.FloatRangeSlider(
                            value=[min_val, max_val],
                            min=min_val,
                            max=max_val,
                            step=0.01,
                            description=name,
                            layout=widgets.Layout(width='80%'),
                            continuous_update=False
                        )
                        is_log = False
                    else:
                        # For other float columns: use logarithmic scale.
                        # Log only makes sense for positive values; fallback to linear if <=0.
                        if min_val <= 0 or max_val <= 0:
                            # fallback to linear
                            slider = widgets.FloatRangeSlider(
                                value=[min_val, max_val],
                                min=min_val,
                                max=max_val,
                                step=0.01,
                                description=name,
                                layout=widgets.Layout(width='80%'),
                                continuous_update=False
                            )
                            is_log = False
                        else:
                            # Compute exponent limits (base 10)
                            min_exp = np.log10(min_val)
                            max_exp = np.log10(max_val)
                            init_low = np.log10(min_val)
                            init_high = np.log10(max_val)
                
                            slider = widgets.FloatRangeSlider(
                                value=[init_low, init_high],
                                min=min_exp,
                                max=max_exp,
                                step=0.01,           # smooth stepping in exponent-space
                                description=name,
                                layout=widgets.Layout(width='80%'),
                                continuous_update=False,
                                readout_format='.2f' # shows exponent on the handle
                            )
                            is_log = True
                
                # Create the output area
                output = widgets.Output()
                
                # Define the observer – it works for both linear and log sliders
                def on_range_change(change, out=output, col=name, log_flag=is_log):
                    with out:
                        out.clear_output()
                        low, high = change['new']
                        if log_flag:
                            # Convert exponent back to actual value
                            actual_low = 10 ** low
                            actual_high = 10 ** high
                            # Show the actual range in the output (optional)
                            print(f"Actual range: {actual_low:.4f} to {actual_high:.4f}")
                        else:
                            actual_low = low
                            actual_high = high
                
                        # Build the snippet using the actual values
                        snippet = f"{col} BETWEEN {actual_low} AND {actual_high}\n"
                        conditions[col] = snippet
                        rebuild_query()
                
                # Attach the observer
                slider.observe(on_range_change, names='value')
                
                # Trigger the initial display (use the correct initial values)
                if is_log:
                    init_change = {'new': [np.log10(min_val), np.log10(max_val)]}
                else:
                    init_change = {'new': [min_val, max_val]}
                on_range_change(init_change)
                
                # Display the slider and its output
                row_widget = widgets.VBox([slider, output])
                display(row_widget)
                print("\n\n\n")

    rebuild_query()

    print("\n\n \t\t\t QUERY FOR THE MOMENT")
    display(full_query_output)

## Código del menú para las gráficas

In [4]:
# Container to display the current list
output_read = widgets.Output()
df = None

def check_values(COLS, data_dict):
    
    cols_set = set(COLS)  # Convert to set for O(1) membership tests
    result = []

    for key, value_list in data_dict.items():
        # Check if every item in value_list exists in cols_set
        if all(item in cols_set for item in value_list):
            result.append(key)

    return result


# Para realizar la QUERY, hacer checks y llamar a las gráficas
def read_data(b):
    
    
    with output_read:

        global graphs, df
        
        output_read.clear_output()
        # Se realiza la SQL query
        df = pd.read_sql_query(QUERY, conn)
    
        # Miramos los datos que tiene
        COLS = list(df.keys())
    
        # Revisamos que las gráficas que se quieren hacer se puedan hacer con las gráficas que tenemos
        possible_graphs = check_values(COLS, dict(checkbox_data))

        flag = all(g in possible_graphs for g in graphs)

        # whatafac why do I have to negate this?????
        if not flag:
            print("ERROR: you don't have enough data to make the graphs")
            print(f"You can make the following graphs")
            print(possible_graphs)
        else: 
            print("Data read. Graphs can be done")

    
# Create the button
button_read_data = widgets.Button(
    description="Read and check data",
    button_style="primary",  # 'primary', 'success', 'info', 'warning', 'danger' or ''
    tooltip="Realiza la Query de SQL y la guarda en un pd.DataFrame. Revisa que los datos tengan las columnas necesarias para hacer las gráficas que se piden",
    layout=widgets.Layout(width='300px')  # Let the button size itself
)

# Attach the function to the button's on_click event
button_read_data.on_click(read_data)

# Center the button using a container box
container = widgets.VBox([button_read_data, output_read])

## Funciones para hacer las gráficas

In [53]:

import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


out_plots = widgets.Output()


# ----------------------------------------------------------------------
# 1. Detection method distribution – bar chart only
def plot_method_distribution(ax, data):
    counts = data['discoverymethod'].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue')
    ax.set_title('Detection Method Distribution')
    ax.set_xlabel('Method')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

# 2. Period–radius diagram – scatter, log‑log
def plot_period_radius(ax, data):
    ax.scatter(data['pl_orbper'], data['pl_rade'], s=8, alpha=0.6)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title('Period–Radius Diagram')
    ax.set_xlabel('Orbital Period (days)')
    ax.set_ylabel('Planet Radius (R$_\oplus$)')

# 3. Mass–radius relation – scatter + linear fit (log‑log)
def plot_mass_radius(ax, data):
    ax.scatter(data['pl_bmasse'], data['pl_rade'], s=8, alpha=0.6)
    ax.set_xscale('log')
    ax.set_yscale('log')
    # Fit power law: log10(R) = a*log10(M) + b
    log_m = np.log10(data['pl_bmasse'])
    log_r = np.log10(data['pl_rade'])
    coeffs = np.polyfit(log_m, log_r, 1)
    x_fit = np.linspace(log_m.min(), log_m.max(), 100)
    y_fit = np.polyval(coeffs, x_fit)
    ax.plot(10**x_fit, 10**y_fit, color='red', lw=2,
            label=f'R ∝ M$^{{{coeffs[0]:.2f}}}$')
    ax.legend()
    ax.set_title('Mass–Radius Relation')
    ax.set_xlabel('Planet Mass (M$_\oplus$)')
    ax.set_ylabel('Planet Radius (R$_\oplus$)')

# 4. Equilibrium temperature histogram by zone – coloured histogram
def plot_temp_histogram(ax, data):
    # Define zones (you can adjust thresholds)
    cold = data[data['pl_eqt'] < 200]
    temperate = data[(data['pl_eqt'] >= 200) & (data['pl_eqt'] <= 800)]
    hot = data[data['pl_eqt'] > 800]
    ax.hist(cold['pl_eqt'], bins=15, alpha=0.6, label='Cold (<200 K)', color='dodgerblue')
    ax.hist(temperate['pl_eqt'], bins=15, alpha=0.6, label='Temperate (200–800 K)', color='limegreen')
    ax.hist(hot['pl_eqt'], bins=15, alpha=0.6, label='Hot (>800 K)', color='tomato')
    ax.set_title('Equilibrium Temperature by Zone')
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('Count')
    ax.legend()

# 5. Boxplots of Rp by host star type – first letter of spectral class
def plot_rp_by_spectype(ax, data):
    # Extract first character (main spectral class)
    data['spec_class'] = data['st_spectype'].str[0].str.upper()
    sns.boxplot(ax=ax, x='spec_class', y='pl_rade', data=data, palette='viridis')
    ax.set_title('Planet Radius by Host Star Type')
    ax.set_xlabel('Spectral Class')
    ax.set_ylabel('Planet Radius (R$_\oplus$)')

# 6. Discovery timeline – cumulative stacked area chart (one per method)
def plot_discovery_timeline(ax, data):
    # Pivot: years as index, methods as columns, counts as values
    grouped = data.groupby(['disc_year', 'discoverymethod']).size().unstack(fill_value=0)
    grouped = grouped.sort_index()
    cum = grouped.cumsum()
    ax.stackplot(cum.index, cum.T, labels=cum.columns, alpha=0.7)
    ax.set_title('Discovery Timeline (Cumulative)')
    ax.set_xlabel('Discovery Year')
    ax.set_ylabel('Cumulative Count')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='x-small')

# ----------------------------------------------------------------------
# Orchestrator: builds a figure from a list of plot names
# ----------------------------------------------------------------------
def create_exoplanet_figure(trash):
    global df, graphs

    with out_plots:
        out_plots.clear_output()
        data = df 
    
        plot_list = graphs
        
        # Map plot names to their functions
        func_map = {
            'Detection method distribution': plot_method_distribution,
            'Period-radius diagram': plot_period_radius,
            'Mass-radius relation': plot_mass_radius,
            'Equilibrium temperature histogram': plot_temp_histogram,
            '$R_p$ by host star type': plot_rp_by_spectype,
            'Discovery timeline': plot_discovery_timeline,
        }
    
        n = len(plot_list)
    
        # Arrange subplots in a grid with 2 columns (except when only 1 plot)
        if n == 1:
            rows, cols = 1, 1
        else:
            cols = 2
            rows = (n + 1) // 2

        fig, axes = plt.subplots(rows, cols, figsize=(14, 18), constrained_layout=True)

        if n == 1:
            ax = axes
            axes = [ax]
        else:
            axes = axes.flatten()

        
        for i, plot_name in enumerate(plot_list):
            if i >= len(axes):
                break
            func = func_map.get(plot_name)
            if func is None:
                axes[i].text(0.5, 0.5, f'Unknown: {plot_name}', ha='center', va='center')
                axes[i].set_axis_off()
            else:
                func(axes[i], data)
    
        # Turn off any unused subplots
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout(pad=3.0)
        plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.92, wspace=0.25, hspace=0.35)
        plt.savefig("plots.pdf")
        plt.show()



# Create the button
button_plot = widgets.Button(
    description="Make plots",
    button_style="primary",  # 'primary', 'success', 'info', 'warning', 'danger' or ''
    layout=widgets.Layout(width='300px')  # Let the button size itself
)

# Attach the function to the button's on_click event
button_plot.on_click(create_exoplanet_figure)

# Center the button using a container box
container_plot = widgets.VBox([button_plot, out_plots])



<>:25: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:42: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:43: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:66: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:25: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:42: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\o"? A raw string is also an option.
<>:43: SyntaxWarning: "\o" is an invalid escape sequence. Such sequences wil

# Menú



## Elegir las gráficas que se van a realizar

Aquí elegimos, a priori, las columnas que vamos a utilizar en el query

In [6]:
display(ui)
display(ui_save)

## Menú de carga

Aquí cargamos la base de datos que vamos a utilizar

In [7]:
display(ui_load)

## Menú de filtros específicos

Por cada una de las tablas que tengamos, vamos a crear unos filtros especiales que le funcionen sólo y únicamente a esas tablas. 
En el caso de columnas de tipo texto, vamos a tener un menú donde podemos elegir qué opciones mantener y cuáles no. 
Para la columna `spec_type`, las opciones serán únicamente la primera letra del tipo de emisión.
Para datos numéricos, se tiene un slider con un rango.

In [62]:
create_filters()

						 DISCOVERYMETHOD 




Output()





						 PL_ORBPER 








						 PL_RADE 








						 PL_BMASSE 




...





						 ST_SPECTYPE 




Output()





						 DISC_YEAR 










 			 QUERY FOR THE MOMENT


Output()

## Menú para hacer las gráficas

Básicamente me robo el primer menú para mostrar las gráficas a hacer. Si no se pueden hacer las gráficas con el database seleccionado, entonces se dice error.

In [16]:
display(ui)
display(container)

In [54]:
display(container_plot)